<!--nav--> [🗺 Learning path](README.md) · **21/43** · ◀ [Simple MultiGPU Audio](./Simple_MultiGPU_Audio.ipynb) · [The Serving Playbook (start here)](./The_Serving_Playbook.ipynb) ▶

# From Fine-Tune to Production: The Complete Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/From_FineTune_To_Production.ipynb)

This repo has two halves that, until now, never met. Notebooks 1–20 **train** models. Notebooks
21–40 **serve** them. Nothing connected the two — so the most common real question had no answer:

> "I fine-tuned a model in [notebook 5](./LoRA_QLoRA_FineTuning.ipynb). **Now what?**"

This notebook is that answer: the whole path from a training artifact to a monitored production
endpoint, with a **gate at every stage**. The gates are the point. Each step between your fine-tune
and your users can silently degrade quality or performance, and a pipeline without gates is just a
fast way to ship regressions.

```
  fine-tune ──► export ──► ✅gate ──► quantize ──► ✅gate ──► serve ──► ✅gate ──► monitor
  (nb 1-20)     Part 1     Part 2      Part 3      Part 3     Part 4    Part 5    Part 6
                           quality               quality-              SLO
                           baseline              after-quant           met?
```

| Part | What you'll do |
|---|---|
| **1** | Export: merge the adapter, or serve it as an adapter — and when each is right |
| **2** | **The quality gate** — build the regression harness *before* you optimize anything |
| **3** | Quantize, then **re-run the gate** — and watch it catch a bad quantization |
| **4** | Serve: derive the config from capacity math instead of guessing |
| **5** | The SLO gate: benchmark before you route real traffic to it |
| **6** | Monitor and roll back |
| **7** | The whole thing as one runnable script, and the checklist |

**Runs on:** any CPU — the pipeline, the gates and the failure modes are all simulated end to end.
GPU-gated cells run the real thing.

In [ ]:
import math, json, random, uuid, statistics
from collections import defaultdict
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 0 · The artifact contract

Everything downstream depends on knowing *exactly* what came out of training. Most production
incidents traced back to fine-tuning start here — a tokenizer that didn't travel with the weights, a
chat template that changed, a base model that was silently a different revision.

**Pin all of this with the artifact, or you will debug it later:**

| Field | Why it matters downstream |
|---|---|
| `base_model` + **revision/commit** | "latest" moves. Your adapter is bonded to a specific base |
| `adapter_path` / `merged_path` | which artifact actually ships |
| `lora_rank`, `target_modules` | must satisfy the server's `--max-lora-rank` (nb 31) |
| **tokenizer** + `chat_template` | a template mismatch silently degrades every response |
| `dtype` used in training | bf16 weights on a T4 (no bf16) is a real trap (nb 33) |
| `eval_baseline` | the numbers the quality gate compares against |
| training data hash | reproducibility, and provenance when quality shifts |

In [ ]:
import json, math, random, statistics, hashlib
from dataclasses import dataclass, field, asdict

@dataclass
class Artifact:
    name: str
    base_model: str
    base_revision: str
    kind: str                      # "lora_adapter" | "merged" | "quantized"
    path: str
    dtype: str = "bfloat16"
    lora_rank: int | None = None
    target_modules: tuple = ()
    chat_template_sha: str = ""
    quantization: str | None = None
    eval_baseline: dict = field(default_factory=dict)
    notes: str = ""

    def summary(self):
        return (f"{self.name} [{self.kind}"
                + (f"/{self.quantization}" if self.quantization else "") + "]")

def template_sha(template: str) -> str:
    return hashlib.sha256(template.encode()).hexdigest()[:12]

CHAT_TEMPLATE = "{% for m in messages %}<|im_start|>{{m['role']}}\n{{m['content']}}<|im_end|>\n{% endfor %}"

trained = Artifact(
    name="support-assistant-v3",
    base_model="Qwen/Qwen2.5-0.5B-Instruct",
    base_revision="a8b602d",                      # pin it - "main" moves under you
    kind="lora_adapter",
    path="/artifacts/support-v3-lora",
    dtype="bfloat16",
    lora_rank=16,
    target_modules=("q_proj", "k_proj", "v_proj", "o_proj"),
    chat_template_sha=template_sha(CHAT_TEMPLATE),
    notes="trained per notebook 5 (LoRA/QLoRA fine-tuning)",
)
print(json.dumps(asdict(trained), indent=2, default=str))

# The checks that cost nothing now and hours later.
def preflight(a: Artifact, server_max_lora_rank=16, gpu_supports_bf16=True):
    problems = []
    if a.base_revision in ("", "main", "master", "latest"):
        problems.append("base_revision is not pinned - the base model can change under your adapter")
    if a.kind == "lora_adapter" and a.lora_rank and a.lora_rank > server_max_lora_rank:
        problems.append(f"lora_rank {a.lora_rank} > server --max-lora-rank {server_max_lora_rank} (nb 31)")
    if a.dtype == "bfloat16" and not gpu_supports_bf16:
        problems.append("artifact is bf16 but the target GPU has no bf16 (Turing/T4) - see nb 33")
    if not a.chat_template_sha:
        problems.append("no chat template recorded - a template mismatch degrades every response")
    return problems

print("\npreflight against a T4 (no bf16), server --max-lora-rank 8:")
for p in preflight(trained, server_max_lora_rank=8, gpu_supports_bf16=False):
    print("  ❌", p)
print("\npreflight against an A100, server --max-lora-rank 16:")
print("  ✅ clean" if not preflight(trained) else preflight(trained))

## Part 1 · Export: merge, or serve the adapter?

Two paths out of a LoRA fine-tune, and the decision is a *serving* decision, not a training one:

| | **Merge into the base** | **Serve as an adapter** |
|---|---|---|
| Artifact | full-size weights (GBs) | a few MB (nb 31) |
| Inference cost | identical to the base model | small per-step overhead |
| Quantization | quantize the merged model normally | quantize the base; adapters stay fp16 |
| Multi-tenant | one deployment per fine-tune | **many fine-tunes, one GPU** |
| Best when | a single model, max throughput | many customers/variants |

**Rule of thumb:** one fine-tune that serves all your traffic → **merge**. More than a couple of
variants → **adapters** (notebook 31's economics are decisive: 50 tenants on one GPU instead of 50).

A subtlety worth knowing: **merging then quantizing is not the same as quantizing then merging.**
Quantize the base and bolt on an fp16 adapter and the adapter's deltas are computed against weights
that no longer exist. Always **merge in fp16/bf16 first, then quantize the merged result**, and
re-run your quality gate afterwards (Part 3).

In [ ]:
def plan_export(n_variants: int, traffic_share_top: float, needs_max_throughput: bool):
    if n_variants == 1:
        return "merge", "single variant - merge for zero serving overhead"
    if n_variants <= 3 and traffic_share_top > 0.9 and needs_max_throughput:
        return "merge (top variant) + adapters (tail)", \
               "one dominant variant justifies a merged deployment; serve the rest as adapters"
    return "adapters", f"{n_variants} variants - multi-LoRA keeps them on one GPU (nb 31)"

for n, share, thr in [(1, 1.0, True), (3, 0.95, True), (12, 0.4, False), (60, 0.1, False)]:
    choice, why = plan_export(n, share, thr)
    print(f"{n:>3} variant(s), top={share:.0%} traffic -> {choice:<34} {why}")

# The merged artifact that continues down the pipeline.
merged = Artifact(**{**asdict(trained), "name": "support-assistant-v3-merged",
                     "kind": "merged", "path": "/artifacts/support-v3-merged",
                     "lora_rank": None, "target_modules": ()})
print(f"\nexported: {merged.summary()}  <- carries the SAME base_revision and chat template")

## Part 2 · The quality gate (build this first)

**This is the part teams skip, and it is the reason "the fine-tune was fine in the notebook but
worse in production" is such a common sentence.**

Before any optimization touches the model, freeze a **regression harness**: fixed prompts, fixed
decoding parameters, and a scoring function. Its output is the baseline every later stage must
match. Three properties matter:

1. **Deterministic** — `temperature=0`, fixed seed. A gate that flickers gets ignored, and an
   ignored gate is worse than none.
2. **Task-shaped** — test what your fine-tune was *for*. Generic benchmarks will not notice that
   your support assistant forgot your refund policy.
3. **Two-sided** — check the fine-tuned behaviour *and* that general capability didn't collapse
   (catastrophic forgetting is real, and quantization interacts with it).

In [ ]:
# A regression harness. Scoring is deliberately simple and auditable - you can read every rule.
EVAL_SUITE = [
    # (id, prompt, must_contain, must_not_contain, category)
    ("refund_window", "What is our refund window?", ["30 day"], ["sorry", "cannot"], "task"),
    ("escalate",      "The customer is furious about a double charge.",
                      ["escalat"], [], "task"),
    ("tone",          "Customer says: this product is garbage.", ["apolog"], ["garbage"], "task"),
    ("policy_no",     "Can I get a refund after 6 months?", ["no", "outside"], ["yes, of course"], "task"),
    ("general_math",  "What is 17 + 25?", ["42"], [], "general"),
    ("general_fact",  "What is the capital of Japan?", ["tokyo"], [], "general"),
    ("general_code",  "Write a python function that returns the square of x.",
                      ["def", "return"], [], "general"),
    ("format_json",   "Reply with JSON containing key 'status'.", ["{", "status"], [], "format"),
]

import re

def contains(haystack, needle):
    # Two failure modes that make eval harnesses lie, both handled here:
    #  1. plain substring matching passes WRONG answers - "no" is inside "processing that now",
    #     so a gate looking for "no" accepts an answer that said yes;
    #  2. strict word-boundary matching breaks INTENTIONAL stems - "escalat" (escalate/
    #     escalated/escalation) and "30 day" (to catch "30 days") would both fail.
    # So short single tokens match on word boundaries; longer needles and phrases are substrings.
    if len(needle) <= 4 and " " not in needle:
        return re.search(rf"(?<!\w){re.escape(needle)}(?!\w)", haystack) is not None
    return needle in haystack

def score_response(case, text):
    _id, prompt, must, must_not, cat = case
    t = text.lower()
    return (all(contains(t, m.lower()) for m in must)
            and not any(contains(t, m.lower()) for m in must_not))

# Prove the matcher does what we claim before trusting any score built on it.
assert contains("no, that is outside our window", "no")          # real "no"
assert not contains("yes, processing that now", "no")            # NOT the "no" in "now"
assert contains("i will escalate this", "escalat")               # stem still works
assert contains("refund window is 30 days", "30 day")            # plural still works
print("matcher self-check passed\n")

def run_eval(model_fn, suite=EVAL_SUITE):
    per_case, by_cat = {}, {}
    for case in suite:
        text = model_fn(case[1])
        ok = score_response(case, text)
        per_case[case[0]] = ok
        by_cat.setdefault(case[4], []).append(ok)
    return {"per_case": per_case,
            "overall": sum(per_case.values()) / len(per_case),
            "by_category": {c: sum(v) / len(v) for c, v in by_cat.items()}}

# --- a simulated model, so the gate itself can be demonstrated without a GPU ---
GOOD_ANSWERS = {
    "What is our refund window?": "Our refund window is 30 days from delivery.",
    "The customer is furious about a double charge.": "I will escalate this to billing immediately.",
    "Customer says: this product is garbage.": "I apologize for the poor experience.",
    "Can I get a refund after 6 months?": "No, that is outside our 30 day window.",
    "What is 17 + 25?": "42",
    "What is the capital of Japan?": "Tokyo",
    "Write a python function that returns the square of x.": "def square(x):\n    return x * x",
    "Reply with JSON containing key 'status'.": '{"status": "ok"}',
}

def make_model(damage=None, seed=0):
    '''damage: None | "forgetting" | "format" | "task" - simulates realistic degradations.'''
    rng = random.Random(seed)
    def fn(prompt):
        ans = GOOD_ANSWERS[prompt]
        if damage == "forgetting" and prompt in ("What is 17 + 25?", "What is the capital of Japan?",
                                                 "Write a python function that returns the square of x."):
            return "I'm here to help with support questions."
        if damage == "format" and prompt.startswith("Reply with JSON"):
            return "Sure! Here is the status: ok"
        if damage == "task" and prompt == "Can I get a refund after 6 months?":
            return "Yes, of course! I can process that refund for you."
        return ans
    return fn

baseline = run_eval(make_model())
merged.eval_baseline = baseline
print("BASELINE (the merged fine-tune, before any optimization)")
print(f"  overall: {baseline['overall']:.0%}")
for cat, v in baseline["by_category"].items():
    print(f"    {cat:<9}{v:.0%}")
print("\nThis is now the contract. Every later stage must match it within tolerance.")

In [ ]:
# The gate itself: compare a candidate against the baseline and decide pass/fail.
def quality_gate(baseline, candidate, overall_tol=0.05, category_tol=0.10, protect=("task",)):
    failures = []
    d_overall = candidate["overall"] - baseline["overall"]
    if d_overall < -overall_tol:
        failures.append(f"overall dropped {abs(d_overall):.0%} (tolerance {overall_tol:.0%})")
    for cat, base_v in baseline["by_category"].items():
        cand_v = candidate["by_category"].get(cat, 0.0)
        tol = 0.0 if cat in protect else category_tol      # protected categories may not regress at all
        if cand_v - base_v < -tol:
            failures.append(f"category '{cat}' dropped {base_v:.0%} -> {cand_v:.0%}"
                            + (" (protected: zero tolerance)" if cat in protect else ""))
    regressed = [k for k, v in baseline["per_case"].items()
                 if v and not candidate["per_case"].get(k, False)]
    if regressed:
        failures.append(f"cases that used to pass now fail: {', '.join(regressed)}")
    return (not failures), failures

print("Running the gate against three realistically-damaged candidates:\n")
for label, damage in [("healthy build", None),
                      ("catastrophic forgetting", "forgetting"),
                      ("format regression", "format"),
                      ("task regression", "task")]:
    cand = run_eval(make_model(damage))
    ok, failures = quality_gate(baseline, cand)
    print(f"{'✅ PASS' if ok else '❌ FAIL'}  {label:<26}overall {cand['overall']:.0%}")
    for f in failures:
        print(f"          - {f}")

print("\nNote the 'format regression' case: overall only drops one case out of eight, so an")
print("aggregate-only gate would wave it through. The per-case check is what catches it -")
print("and in production that one case is every structured-output call you make (nb 29).")

## Part 3 · Quantize, then re-gate

Now — and **only** now, with a baseline in hand — optimize. Quantization (notebook 24) is usually
the first move, and it is exactly the kind of change that needs a gate: it makes the model faster
and *slightly different*, and "slightly different" is invisible until it isn't.

In [ ]:
# Simulated quantization outcomes, from benign to the one that ships broken.
QUANT_PROFILES = {
    "fp16 (no quantization)": dict(damage=None,        bytes_per_weight=2.0, speedup=1.00),
    "fp8":                    dict(damage=None,        bytes_per_weight=1.0, speedup=1.35),
    "int4 AWQ (good)":        dict(damage=None,        bytes_per_weight=0.5, speedup=1.75),
    "int4 (bad calibration)": dict(damage="format",    bytes_per_weight=0.5, speedup=1.75),
    "int4 (over-aggressive)": dict(damage="forgetting", bytes_per_weight=0.5, speedup=1.80),
}

print(f"{'candidate':<26}{'weights':>9}{'speedup':>9}{'quality':>9}   gate")
print("-" * 76)
approved = []
for name, prof in QUANT_PROFILES.items():
    cand = run_eval(make_model(prof["damage"]))
    ok, failures = quality_gate(baseline, cand)
    weights_gb = 0.5 * prof["bytes_per_weight"]        # 0.5B params
    if ok:
        approved.append((name, prof, cand))
    print(f"{name:<26}{weights_gb:>7.2f}GB{prof['speedup']:>8.2f}x{cand['overall']:>9.0%}   "
          f"{'✅ approved' if ok else '❌ REJECTED: ' + failures[0][:38]}")

best = max(approved, key=lambda x: x[1]["speedup"])
print(f"\nFastest candidate that PASSED the gate: {best[0]} ({best[1]['speedup']:.2f}x)")
print("The over-aggressive int4 is 1.80x - and it silently lost general knowledge.")
print("Without the gate you would have shipped it, because throughput dashboards look great.")

quantized = Artifact(**{**asdict(merged), "name": merged.name + "-awq",
                        "kind": "quantized", "quantization": "awq_int4",
                        "path": "/artifacts/support-v3-awq"})
quantized.eval_baseline = best[2]
print(f"\npromoted artifact: {quantized.summary()}")

## Part 4 · Serve: derive the config, don't guess it

You have an approved artifact. The serving configuration should now be **computed** from the
capacity math in notebooks 22 and 27 — not copied from a blog post.

In [ ]:
def serving_config(params_b, weight_bytes, layers, kv_heads, head_dim,
                   gpu_vram_gb, p99_prompt_tokens, target_concurrency,
                   gpu_util=0.90, kv_bytes=2):
    weights_gb = params_b * weight_bytes
    overhead_gb = 1.5                                   # activations + CUDA graphs (nb 27)
    pool_gb = gpu_vram_gb * gpu_util - weights_gb - overhead_gb
    kv_per_token = 2 * layers * kv_heads * head_dim * kv_bytes
    # max_model_len should cover the p99 prompt plus generation headroom, not the model's maximum
    max_model_len = 1 << max(9, math.ceil(math.log2(p99_prompt_tokens * 1.5)))
    pool_tokens = int(pool_gb * 1e9 // kv_per_token) if pool_gb > 0 else 0
    concurrency = pool_tokens // max_model_len if max_model_len else 0
    return {
        "feasible": pool_gb > 0 and concurrency >= 1,
        "weights_gb": round(weights_gb, 2),
        "kv_pool_gb": round(pool_gb, 2),
        "kv_per_token_kb": round(kv_per_token / 1024, 1),
        "max_model_len": max_model_len,
        "kv_pool_tokens": pool_tokens,
        "max_concurrency": concurrency,
        "max_num_seqs": max(1, min(target_concurrency, concurrency)),
        "gpu_memory_utilization": gpu_util,
    }

cfg = serving_config(params_b=0.5, weight_bytes=0.5, layers=24, kv_heads=2, head_dim=64,
                     gpu_vram_gb=16, p99_prompt_tokens=1800, target_concurrency=64)
for k, v in cfg.items():
    print(f"  {k:<26}{v}")

print(f"\n  vllm serve {quantized.path} \\")
print(f"    --quantization awq --dtype half \\")
print(f"    --max-model-len {cfg['max_model_len']} \\")
print(f"    --max-num-seqs {cfg['max_num_seqs']} \\")
print(f"    --gpu-memory-utilization {cfg['gpu_memory_utilization']}")

print("\nThe two numbers people get wrong, both derived above:")
print(f"  --max-model-len : sized to the p99 PROMPT (1800) x1.5, not the model's maximum.")
print(f"                    Serving 32k here would cut concurrency ~{32768//cfg['max_model_len']}x for no benefit (nb 27).")
print(f"  --max-num-seqs  : capped at what KV can actually hold ({cfg['max_concurrency']}), not a round number.")

## Part 5 · The SLO gate

The model passes quality. The server starts. **You are still not done** — the last gate asks whether
this deployment can carry your traffic at your latency promise (notebooks 28 and 40).

In [ ]:
def slo_checks(measured, slo):
    # Pure: returns [(name, measured, target, unit, ok)] so it can be printed OR composed.
    spec = [("TTFT p95",   "ttft_p95_s",  "ttft_p95_s",  "s",   "lower"),
            ("TPOT p95",   "tpot_p95_ms", "tpot_p95_ms", "ms",  "lower"),
            ("goodput",    "goodput_rps", "peak_rps",    "rps", "higher"),
            ("error rate", "error_rate",  "max_error",   "",    "lower")]
    out = []
    for name, mkey, skey, unit, direction in spec:
        got, want = measured[mkey], slo[skey]
        ok = got <= want if direction == "lower" else got >= want
        out.append((name, got, want, unit, ok))
    return out

def slo_gate(measured, slo):
    checks = slo_checks(measured, slo)
    failures = [f"{n}: {g}{u} vs target {w}{u}" for n, g, w, u, ok in checks if not ok]
    return (not failures), failures

def print_slo(measured, slo):
    print(f"{'check':<12}{'measured':>12}{'target':>12}   verdict")
    print("-" * 52)
    for name, got, want, unit, ok in slo_checks(measured, slo):
        print(f"{name:<12}{got:>10.3g}{unit:<2}{want:>10.3g}{unit:<2}   {'✅' if ok else '❌'}")
    return slo_gate(measured, slo)

SLO = dict(ttft_p95_s=1.0, tpot_p95_ms=50.0, peak_rps=12.0, max_error=0.001)

print("Candidate deployment, benchmarked open-loop at peak traffic (nb 28):\n")
measured = dict(ttft_p95_s=0.62, tpot_p95_ms=31.0, goodput_rps=14.4, error_rate=0.0002)
ok, failures = print_slo(measured, SLO)
print(f"\n{'✅ SLO GATE PASSED - safe to route traffic' if ok else '❌ SLO GATE FAILED'}")

print("\n\nThe same deployment with one replica lost (the N-1 test from nb 38):\n")
degraded = dict(ttft_p95_s=3.9, tpot_p95_ms=44.0, goodput_rps=9.6, error_rate=0.0004)
ok2, failures2 = print_slo(degraded, SLO)
print(f"\n{'✅' if ok2 else '❌'} N-1 capacity: " +
      ("survives a replica failure" if ok2 else "would breach SLO if one replica dies"))
for f in failures2:
    print(f"     - {f}")
print("\nShip the first, but size the fleet for the second. A deployment that only meets SLO")
print("at full health is one node failure away from an incident (nb 38 Part 6).")

## Part 6 · Monitor and roll back

The pipeline doesn't end at deploy. From [notebook 27](./Serving_Logs_Observability.ipynb), the
signals that specifically matter for a *freshly deployed fine-tune*:

| Watch | Why it's different after a model change |
|---|---|
| `gpu_cache_usage_perc` | a new artifact changes weight size → changes the KV pool |
| startup log "Model loading took … GiB" | the fastest confirmation your quantized artifact actually loaded |
| prefix cache hit rate | a changed chat template invalidates every cached prefix |
| `finished_reason="length"` | template or stop-token drift shows up here first |
| TTFT / TPOT p95 | compare to the pre-deploy benchmark, not to "feels fine" |
| the quality gate, **on a live sample** | offline eval never fully covers production traffic |

**Rollback should be a decision you already made.** Write the trigger down before you deploy:

In [ ]:
ROLLBACK_TRIGGERS = [
    ("TTFT p95 > 2x pre-deploy baseline for 10 min", "automatic"),
    ("error rate > 1%", "automatic"),
    ("KV usage > 95% sustained (new artifact is bigger than planned)", "automatic"),
    ("prefix cache hit rate collapses >50% (template changed)", "page a human"),
    ("live-sample quality gate fails", "page a human"),
]
print("Rollback policy (write this BEFORE deploying):\n")
for trigger, action in ROLLBACK_TRIGGERS:
    print(f"  {action:<14}{trigger}")

print("\nAnd the rollback itself is just the pipeline in reverse - which is only fast if you")
print("kept the previous artifact addressable:\n")
print("  1. route traffic back to the previous artifact  (keep N-1 versions warm, nb 38)")
print("  2. drain the new replicas gracefully            (nb 38 Part 4)")
print("  3. keep the failed artifact for diagnosis       - do NOT delete it")
print("  4. re-run the quality gate offline against production samples you captured")

# The full pipeline, as a state machine you can actually execute.
def run_pipeline(artifact, quant_profile, measured, slo, verbose=True):
    stages = []
    def stage(name, ok, detail):
        stages.append({"stage": name, "ok": ok, "detail": detail})
        if verbose:
            print(f"  {'✅' if ok else '❌'} {name:<22}{detail}")
        return ok

    if verbose: print(f"pipeline for {artifact.summary()}\n")
    problems = preflight(artifact)
    if not stage("preflight", not problems, problems[0] if problems else "artifact contract complete"):
        return stages, False
    base = run_eval(make_model())
    if not stage("baseline eval", True, f"overall {base['overall']:.0%}"):
        return stages, False
    cand = run_eval(make_model(quant_profile["damage"]))
    ok, fails = quality_gate(base, cand)
    if not stage("quality gate", ok, f"overall {cand['overall']:.0%}"
                 + ("" if ok else f" - {fails[0][:44]}")):
        return stages, False
    cfg = serving_config(0.5, quant_profile["bytes_per_weight"], 24, 2, 64, 16, 1800, 64)
    if not stage("capacity plan", cfg["feasible"],
                 f"max_model_len {cfg['max_model_len']}, {cfg['max_concurrency']} concurrent"):
        return stages, False
    ok, fails = slo_gate(measured, slo)
    if not stage("SLO gate", ok, f"TTFT p95 {measured['ttft_p95_s']}s, goodput {measured['goodput_rps']} rps"
                 + ("" if ok else f" - {fails[0][:40]}")):
        return stages, False
    stage("deploy", True, "canary 1 replica, then roll forward")
    return stages, True

print("\n" + "=" * 70)
print("HAPPY PATH")
print("=" * 70)
_, ok = run_pipeline(quantized, QUANT_PROFILES["int4 AWQ (good)"], measured, SLO)
print(f"\nresult: {'SHIPPED' if ok else 'BLOCKED'}")

print("\n" + "=" * 70)
print("THE PIPELINE DOING ITS JOB (over-aggressive quantization)")
print("=" * 70)
_, ok = run_pipeline(quantized, QUANT_PROFILES["int4 (over-aggressive)"], measured, SLO)
print(f"\nresult: {'SHIPPED' if ok else 'BLOCKED - regression never reached users'}")

## Part 7 · The real thing

The cell below runs the whole pipeline for real on a GPU: trains a tiny adapter, merges it,
evaluates it, serves it with vLLM, queries it, and reports. It is deliberately small enough to
finish on a free T4 — the point is the **shape**, which is identical at production scale.

The repo also ships this as a script: `python tools/e2e_pipeline.py --help`.

In [ ]:
# GPU-ONLY end-to-end run: train -> merge -> gate -> serve -> query -> verdict.
import torch
if not torch.cuda.is_available():
    print("No GPU - the simulated pipeline above already demonstrated every stage and gate.")
    print("On a T4 this cell runs the real thing (~10 min), or use:")
    print("    python tools/e2e_pipeline.py --model Qwen/Qwen2.5-0.5B-Instruct")
else:
    import os, time, subprocess, urllib.request
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model

    BASE = "Qwen/Qwen2.5-0.5B-Instruct"
    OUT = "/content/e2e"
    os.makedirs(OUT, exist_ok=True)

    # 1. TRAIN a tiny adapter (the notebook 5 step, compressed)
    tok = AutoTokenizer.from_pretrained(BASE)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    PAIRS = [("What is our refund window?", "Our refund window is 30 days from delivery."),
             ("Can I get a refund after 6 months?", "No, that is outside our 30 day window."),
             ("The customer is furious about a double charge.",
              "I will escalate this to billing immediately.")]
    model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32).to("cuda")
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0,
                                             target_modules=["q_proj","k_proj","v_proj","o_proj"],
                                             task_type="CAUSAL_LM"))
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
    texts = [tok.apply_chat_template([{"role":"user","content":q},{"role":"assistant","content":a}],
                                     tokenize=False) for q, a in PAIRS]
    model.train()
    for _ in range(40):
        enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to("cuda")
        loss = model(**enc, labels=enc.input_ids).loss
        loss.backward(); opt.step(); opt.zero_grad()
    print(f"1. trained  · final loss {loss.item():.3f}")

    # 2. MERGE (so we can serve it as a plain model)
    merged_dir = f"{OUT}/merged"
    model.merge_and_unload().save_pretrained(merged_dir)
    tok.save_pretrained(merged_dir)
    del model, opt; torch.cuda.empty_cache()
    print(f"2. merged   · {merged_dir}")

    # 3. SERVE
    server = subprocess.Popen(
        ["vllm", "serve", merged_dir, "--dtype", "half", "--max-model-len", "1024",
         "--gpu-memory-utilization", "0.80", "--max-num-seqs", "16", "--port", "8000",
         "--served-model-name", "support-v3"],
        stdout=open(f"{OUT}/server.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(180):
        try: urllib.request.urlopen("http://localhost:8000/health", timeout=2); break
        except Exception: time.sleep(2)
    print("3. serving  · vLLM up")

    # 4. GATE against the live endpoint
    from openai import OpenAI
    client = OpenAI(base_url="http://localhost:8000/v1", api_key="x")
    def live(prompt):
        return client.chat.completions.create(model="support-v3", temperature=0.0, max_tokens=60,
                                              messages=[{"role":"user","content":prompt}]
                                              ).choices[0].message.content
    LIVE_SUITE = [c for c in EVAL_SUITE if c[4] in ("task", "general")]
    live_result = run_eval(live, LIVE_SUITE)
    print(f"4. gate     · live overall {live_result['overall']:.0%}")
    for cid, ok in live_result["per_case"].items():
        print(f"              {'✅' if ok else '❌'} {cid}")

    # 5. VERDICT
    print("\n5. verdict  · " + ("SHIPPABLE" if live_result["overall"] >= 0.6
                                else "BLOCKED - a 40-step toy fine-tune is expected to be weak here;"
                                     " the GATE working is the point, not the score"))
    server.terminate(); server.wait(timeout=20)
    print("   server stopped")

## Part 8 · The end-to-end checklist

**Artifact**
- [ ] `base_model` **and revision** pinned; tokenizer + chat template shipped with the weights
- [ ] `lora_rank` ≤ the server's `--max-lora-rank`; dtype supported by the target GPU (nb 33)
- [ ] Export decision made deliberately: merge vs adapters (Part 1, nb 31)

**Quality**
- [ ] Regression suite written **before** optimizing, deterministic, task-shaped
- [ ] Baseline recorded *with the artifact*
- [ ] Gate protects task categories at **zero tolerance**, and checks per-case, not just aggregate
- [ ] Re-run after **every** transformation: merge, quantize, kernel/engine upgrade

**Serving**
- [ ] `--max-model-len` derived from the **p99 prompt**, not the model's maximum
- [ ] `--max-num-seqs` capped by what KV can hold (Part 4, nb 27)
- [ ] Startup log read once and compared to the plan (nb 27 Part 1)

**Before traffic**
- [ ] Open-loop benchmark at peak, percentiles not means (nb 28)
- [ ] SLO gate passed at **N−1** capacity, not just full health (nb 38)
- [ ] Canary one replica first; alerts on KV usage and queue depth (nb 27)

**After**
- [ ] Rollback triggers written down before deploy, previous artifact kept warm
- [ ] Live-sample quality gate on a schedule, not just at deploy time

## Recap

1. **The gap between "trained" and "served" is where quality quietly dies.** Every stage —
   merging, quantizing, templating, config — can degrade the model, and none of them raise an error.
2. **Build the quality gate first**, before any optimization. It is the contract everything else is
   measured against.
3. **Check per-case, not just aggregate.** The format regression in Part 2 costs one case out of
   eight and every structured-output call in production.
4. **Merge in full precision, then quantize** — never the reverse.
5. **Derive the serving config** from capacity math; the two flags people guess are the two that
   decide your concurrency.
6. **Gate on SLO at N−1**, not at full health.
7. **Decide rollback before you deploy.**

### Where this sits in the repo

| | |
|---|---|
| Comes from | [5 LoRA/QLoRA](./LoRA_QLoRA_FineTuning.ipynb) · [10 Post-Training](./PostTraining_Core_Understanding.ipynb) · [18 Multimodal LoRA](./Multimodal_LoRA_QLoRA_DPO.ipynb) |
| Uses | [24 Quantization](./Quantized_Serving_Showdown.ipynb) · [23 vLLM](./vLLM_High_Throughput_Serving.ipynb) · [27 Observability](./Serving_Logs_Observability.ipynb) · [28 Capacity](./Serving_Benchmark_Capacity_Planning.ipynb) · [31 Multi-LoRA](./MultiLoRA_Serving_At_Scale.ipynb) |
| Then read | [21 The Serving Playbook](./The_Serving_Playbook.ipynb) when something goes wrong · [40 The Optimization Stack](./The_Optimization_Stack.ipynb) before stacking optimizations |

🏁 **This is the end-to-end path.** [Back to the learning path](README.md)